# Machine Learning Assignment 2: Activity Classification Lab

This notebook is a readable walkthrough of the repository workflow. It uses the UCI Human Activity Recognition Using Smartphones dataset and evaluates five required classifiers on the same official test partition.

## Workflow

1. Prepare the public UCI data while preserving its official train/test split.
2. Train and save Logistic Regression, Decision Tree, KNN, Gaussian Naive Bayes, and SVM.
3. Evaluate every saved model on the untouched test data using the six assignment metrics.
4. Inspect the comparison table and run the Streamlit app for interactive evaluation.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / 'src').exists(), 'Open this notebook from the repository root or notebooks folder.'
sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

Project root: /Users/prashant.katariya/Documents/Codex/2026-11-05/you-are-a-student-of-bits/ML-Classfication


## 1. Prepare data

The script downloads the public UCI archive when needed, converts its files into CSV format, and keeps the original 7,352-row training partition and 2,947-row test partition separate. The generated `test_data.csv` contains the 561 required input columns plus the `activity` target column.

In [2]:
from src.prepare_data import main as prepare_data

if not (PROJECT_ROOT / 'data' / 'train_data.csv').exists():
    prepare_data()
else:
    print('Prepared training and test CSV files already exist.')

Prepared training and test CSV files already exist.


In [3]:
import json
import pandas as pd

with (PROJECT_ROOT / 'data' / 'metadata.json').open(encoding='utf-8') as source:
    metadata = json.load(source)
test_data = pd.read_csv(PROJECT_ROOT / 'data' / 'test_data.csv')

print(f"Features: {metadata['number_of_features']}")
print(f"Training rows: {metadata['training_rows']}")
print(f"Test rows: {metadata['test_rows']}")
print(f"Classes: {', '.join(metadata['class_names'])}")
test_data.head()

Features: 561
Training rows: 7352
Test rows: 2947
Classes: LAYING, SITTING, STANDING, WALKING, WALKING_DOWNSTAIRS, WALKING_UPSTAIRS


,feature_001_tBodyAcc-mean()-X,feature_002_tBodyAcc-mean()-Y,feature_003_tBodyAcc-mean()-Z,feature_004_tBodyAcc-std()-X,feature_005_tBodyAcc-std()-Y,feature_006_tBodyAcc-std()-Z,feature_007_tBodyAcc-mad()-X,feature_008_tBodyAcc-mad()-Y,feature_009_tBodyAcc-mad()-Z,feature_010_tBodyAcc-max()-X,...,feature_553_fBodyBodyGyroJerkMag-skewness(),feature_554_fBodyBodyGyroJerkMag-kurtosis(),"feature_555_angle(tBodyAccMean,gravity)","feature_556_angle(tBodyAccJerkMean),gravityMean)","feature_557_angle(tBodyGyroMean,gravityMean)","feature_558_angle(tBodyGyroJerkMean,gravityMean)","feature_559_angle(X,gravityMean)","feature_560_angle(Y,gravityMean)","feature_561_angle(Z,gravityMean)",activity
0,0.257178,-0.023285,-0.014654,-0.938404,-0.920091,-0.667683,-0.952501,-0.925249,-0.674302,-0.894088,...,-0.330370,-0.705974,0.006462,0.162920,-0.825886,0.271151,-0.720009,0.276801,-0.057978,STANDING
1,0.286027,-0.013163,-0.119083,-0.975415,-0.967458,-0.944958,-0.986799,-0.968401,-0.945823,-0.894088,...,-0.121845,-0.594944,-0.083495,0.017500,-0.434375,0.920593,-0.698091,0.281343,-0.083898,STANDING
2,0.275485,-0.026050,-0.118152,-0.993819,-0.969926,-0.962748,-0.994403,-0.970735,-0.963483,-0.939260,...,-0.190422,-0.640736,-0.034956,0.202302,0.064103,0.145068,-0.702771,0.280083,-0.079346,STANDING
3,0.270298,-0.032614,-0.117520,-0.994743,-0.973268,-0.967091,-0.995274,-0.974471,-0.968897,-0.938610,...,-0.344418,-0.736124,-0.017067,0.154438,0.340134,0.296407,-0.698954,0.284114,-0.077108,STANDING
4,0.274833,-0.027848,-0.129527,-0.993852,-0.967445,-0.978295,-0.994111,-0.965953,-0.977346,-0.938610,...,-0.534685,-0.846595,-0.002223,-0.040046,0.736715,-0.118545,-0.692245,0.290722,-0.073857,STANDING


## 2. Train the five models

The training script uses the same train partition for every model. Scaling is kept inside the relevant saved pipelines, so the Streamlit app applies exactly the same preprocessing during evaluation.

In [4]:
from src.train_models import main as train_models

required_model = PROJECT_ROOT / 'model' / 'support_vector_machine.joblib'
if not required_model.exists():
    train_models()
else:
    print('Five saved model artifacts already exist.')

Five saved model artifacts already exist.


## 3. Evaluate on untouched test data

Accuracy, Precision, Recall, and F1 use predicted labels. Precision, Recall, and F1 are weighted across the six classes. AUC is macro one-vs-rest AUC, and MCC is Matthews Correlation Coefficient. The evaluation script also writes a classification report and one confusion matrix per model.

In [5]:
from src.evaluate_models import main as evaluate_models

evaluate_models()

Evaluating logistic_regression...


Evaluating decision_tree...
Evaluating knn...


/Users/prashant.katariya/Documents/Codex/2026-11-05/you-are-a-student-of-bits/ML-Classfication/.venv/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py:134: UserWarning: Could not find the number of physical cores for the following reason:
invalid literal for int() with base 10: ''
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "/Users/prashant.katariya/Documents/Codex/2026-11-05/you-are-a-student-of-bits/ML-Classfication/.venv/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py", line 264, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_darwin()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/prashant.katariya/Documents/Codex/2026-11-05/you-are-a-student-of-bits/ML-Classfication/.venv/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py", line 420, in _count_physic

Evaluating gaussian_naive_bayes...
Evaluating support_vector_machine...



Evaluation metrics
         ML Model Name  Accuracy    AUC  Precision  Recall     F1    MCC
   Logistic Regression    0.9535 0.9975     0.9552  0.9535 0.9534 0.9445
         Decision Tree    0.8585 0.9211     0.8585  0.8585 0.8579 0.8302
   K-Nearest Neighbors    0.8890 0.9821     0.8965  0.8890 0.8879 0.8683
  Gaussian Naive Bayes    0.7703 0.9578     0.7947  0.7703 0.7688 0.7286
Support Vector Machine    0.9528 0.9937     0.9534  0.9528 0.9527 0.9435

Saved results to /Users/prashant.katariya/Documents/Codex/2026-11-05/you-are-a-student-of-bits/ML-Classfication/results


In [6]:
metrics = pd.read_csv(PROJECT_ROOT / 'results' / 'metrics.csv')
metrics

,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.9535,0.9975,0.9552,0.9535,0.9534,0.9445
1,Decision Tree,0.8585,0.9211,0.8585,0.8585,0.8579,0.8302
2,K-Nearest Neighbors,0.8890,0.9821,0.8965,0.8890,0.8879,0.8683
3,Gaussian Naive Bayes,0.7703,0.9578,0.7947,0.7703,0.7688,0.7286
4,Support Vector Machine,0.9528,0.9937,0.9534,0.9528,0.9527,0.9435


## 4. Overall result and Streamlit app

Logistic Regression is the winner on this test partition because it has the highest Accuracy, AUC, F1, and MCC. To inspect any model interactively without retraining it, run the following command in a terminal from the repository root:

```bash
streamlit run app.py
```

Upload a labelled CSV with the same 561 feature columns and the `activity` target column, or use the included `data/test_data.csv`.